# NumPy para Cómputo de Alto Rendimiento

En los notebooks anteriores aprendimos a crear y manipular arreglos, hacer álgebra lineal y construir pipelines de datos. Todo eso es el **qué**. Este notebook es el **por qué es rápido** y, más importante, el **cómo hacerlo más rápido todavía**.

La premisa central del cómputo de alto rendimiento es simple: el hardware ya es rápido. Lo que generalmente lo frena es el código. Vamos a aprender a no ser el cuello de botella.

Cada sección tiene su propio benchmark para que veas los números en tu máquina, no solo leas que "es más rápido".

Los temas que cubriremos:

1. Herramientas de medición
2. El costo del intérprete de Python — por qué los loops son lentos
3. Vectorización — eliminar loops
4. Memoria y cache: layout C vs Fortran
5. Vistas vs copias — no muevas datos que no tienes que mover
6. Operaciones in-place
7. Selección de dtype
8. Preasignación de memoria
9. `np.einsum` — operaciones tensóricas sin loops ni transpuestas
10. Conectando con el hardware: BLAS y paralelismo implícito

In [ ]:
import numpy as np
import time

print(f"NumPy version: {np.__version__}")

# Configuración de NumPy para que los prints sean legibles
np.set_printoptions(precision=4, suppress=True)

---
# 1. Herramientas de Medición

No puedes optimizar lo que no puedes medir. Antes de tocar una sola línea de código, necesitas saber exactamente cuánto tarda cada parte. Las dos herramientas principales son `%timeit` (magic de Jupyter) y `time.perf_counter` (Python puro).

La diferencia importante:
- `%timeit` corre el código **muchas veces** y reporta el mejor promedio. Úsalo para benchmarks limpios.
- `time.perf_counter` mide una sola ejecución. Úsalo dentro del código para medir bloques específicos.

In [ ]:
arr = np.random.rand(1_000_000)

# %timeit corre el statement en un loop y reporta media ± desviación
%timeit np.sum(arr)

In [ ]:
# time.perf_counter — alta resolución, ideal para bloques de código
arr = np.random.rand(1_000_000)

t0 = time.perf_counter()
resultado = np.sum(arr)
t1 = time.perf_counter()

print(f"Tiempo: {(t1 - t0) * 1000:.4f} ms")

In [ ]:
# Función utilitaria para benchmarks — la usaremos en todo el notebook
def bench(label, func, *args, reps=5):
    """
    Corre `func(*args)` `reps` veces, descarta el primer run (warm-up)
    y reporta el tiempo promedio de los restantes.
    """
    # warm-up
    func(*args)

    tiempos = []
    for _ in range(reps):
        t0 = time.perf_counter()
        func(*args)
        t1 = time.perf_counter()
        tiempos.append(t1 - t0)

    media = np.mean(tiempos) * 1000   # en ms
    print(f"  {label:<40} {media:>8.3f} ms")
    return media

print("Función bench lista ✓")

---
# 2. El Costo del Intérprete de Python

Python es un lenguaje interpretado y dinámicamente tipado. Cada vez que ejecutas una instrucción, el intérprete tiene que:

1. Resolver qué tipo de objeto es cada variable
2. Buscar la operación correspondiente en su tabla de dispatch
3. Ejecutarla
4. Manejar el refcount del garbage collector

Eso cuesta tiempo. Cuando haces un loop de un millón de iteraciones, pagas ese costo **un millón de veces**. NumPy lo paga **una sola vez** por operación vectorizada.

Vamos a verlo con números reales.

In [ ]:
N = 1_000_000
lista = list(range(N))
arr   = np.arange(N, dtype=np.float64)

def suma_loop_python(lst):
    total = 0.0
    for x in lst:
        total += x
    return total

def suma_builtin(lst):
    return sum(lst)        # sum() de Python — sigue siendo Python puro

def suma_numpy(a):
    return np.sum(a)       # implementado en C

print("Suma de 1,000,000 elementos:")
t1 = bench("loop Python explícito",   suma_loop_python, lista)
t2 = bench("sum() builtin de Python", suma_builtin,     lista)
t3 = bench("np.sum() NumPy",          suma_numpy,       arr)

print()
print(f"  NumPy es ~{t2/t3:.0f}x más rápido que sum() builtin")
print(f"  NumPy es ~{t1/t3:.0f}x más rápido que el loop explícito")

Nota que incluso `sum()` builtin de Python es notablemente más lento que NumPy. `sum()` ya está implementado en C internamente, pero sigue operando sobre objetos Python (`PyObject*`) con todo su overhead. NumPy opera directamente sobre los bytes del buffer.

In [ ]:
# El costo escala linealmente con el tamaño — verifiquemos
print("Cómo escala el tiempo con el tamaño del arreglo:")
print(f"  {'N':>12}   {'np.sum (ms)':>12}")
print("  " + "-" * 28)

for exp in [4, 5, 6, 7, 8]:
    n = 10 ** exp
    a = np.random.rand(n)
    t0 = time.perf_counter()
    for _ in range(10):
        np.sum(a)
    t1 = time.perf_counter()
    ms = (t1 - t0) / 10 * 1000
    print(f"  {n:>12,}   {ms:>12.4f}")

---
# 3. Vectorización — Eliminar Loops

"Vectorizar" significa reescribir un loop explícito de Python como una operación sobre arreglos completos. El loop no desaparece — se mueve a C, donde corre órdenes de magnitud más rápido.

La regla es simple: **si puedes expresar lo que hace tu loop como una operación de NumPy, hazlo**.

## 3.1 Operaciones elemento a elemento

In [ ]:
N = 2_000_000
a = np.random.rand(N)
b = np.random.rand(N)

# Versión con loop
def suma_vectores_loop(a, b):
    resultado = np.empty(len(a))
    for i in range(len(a)):
        resultado[i] = a[i] + b[i]
    return resultado

# Versión vectorizada
def suma_vectores_numpy(a, b):
    return a + b

print("Suma elemento a elemento (2M elementos):")
t1 = bench("loop Python",  suma_vectores_loop,  a, b, reps=3)
t2 = bench("a + b NumPy",  suma_vectores_numpy, a, b)
print(f"\n  Speedup: ~{t1/t2:.0f}x")

## 3.2 Funciones matemáticas

In [ ]:
import math

N = 500_000
x_lista = list(np.random.rand(N))
x_arr   = np.array(x_lista)

def sqrt_loop(lst):
    return [math.sqrt(v) for v in lst]

def sqrt_numpy(arr):
    return np.sqrt(arr)

print("sqrt() de 500,000 elementos:")
t1 = bench("list comprehension + math.sqrt", sqrt_loop,  x_lista)
t2 = bench("np.sqrt()",                      sqrt_numpy, x_arr)
print(f"\n  Speedup: ~{t1/t2:.0f}x")

## 3.3 Condicionales — reemplazar `if` dentro de loops

In [ ]:
N = 1_000_000
datos = np.random.randn(N)   # distribución normal

# Versión con loop — aplicar ReLU (max(0, x)) elemento a elemento
def relu_loop(arr):
    resultado = np.empty(len(arr))
    for i in range(len(arr)):
        resultado[i] = arr[i] if arr[i] > 0 else 0
    return resultado

# Versiones vectorizadas — tres formas equivalentes
def relu_clip(arr):    return np.clip(arr, 0, None)
def relu_maximum(arr): return np.maximum(arr, 0)
def relu_where(arr):   return np.where(arr > 0, arr, 0)

print("ReLU en 1M elementos:")
t0 = bench("loop Python con if", relu_loop,    datos, reps=3)
bench("np.clip(arr, 0, None)",   relu_clip,    datos)
bench("np.maximum(arr, 0)",      relu_maximum, datos)
bench("np.where(arr > 0, arr, 0)", relu_where, datos)

t1 = bench("np.maximum (para speedup)", relu_maximum, datos)
print(f"\n  Speedup: ~{t0/t1:.0f}x")

---
# 4. Memoria y Cache: Layout C vs Fortran

Aquí empieza la parte que ya casi nadie explica en cursos introductores, pero es crítica en HPC.

Una matriz 2D en memoria siempre se almacena como un bloque lineal de bytes. La pregunta es **en qué orden** van los elementos:

- **C-order (row-major):** las filas son contiguas → `[fila0col0, fila0col1, ..., fila1col0, ...]`
- **Fortran-order (column-major):** las columnas son contiguas → `[fila0col0, fila1col0, ..., fila0col1, ...]`

NumPy usa C-order por default. Esto importa porque el **CPU prefetcher** carga datos en cache en bloques contiguos (cache lines). Si accedes a memoria en saltos grandes, cada acceso es un **cache miss** — el CPU tiene que esperar a RAM, que es ~100x más lenta que L1 cache.

In [ ]:
# Creamos la misma matriz en C-order y Fortran-order
N = 5000
M_c = np.random.rand(N, N)                    # C-order (default)
M_f = np.asfortranarray(M_c)                  # Fortran-order (mismos datos, distinto layout)

print(f"M_c flags C_CONTIGUOUS: {M_c.flags['C_CONTIGUOUS']}")
print(f"M_c flags F_CONTIGUOUS: {M_c.flags['F_CONTIGUOUS']}")
print()
print(f"M_c strides: {M_c.strides}   ← salto de {M_c.strides[0]} bytes para siguiente fila")
print(f"M_f strides: {M_f.strides}   ← salto de {M_f.strides[1]} bytes para siguiente columna")

In [ ]:
# Benchmark: suma por filas vs suma por columnas en C-order
# En C-order, recorrer filas es cache-friendly, recorrer columnas no

def suma_filas(M):    return np.sum(M, axis=1)   # suma cada fila → recorre por columna dentro de fila
def suma_columnas(M): return np.sum(M, axis=0)   # suma cada columna → salta N*8 bytes entre elementos

print(f"Suma sobre matriz {N}x{N} C-order:")
t_filas = bench("np.sum(M, axis=1)  [filas — cache-friendly]",    suma_filas,    M_c)
t_cols  = bench("np.sum(M, axis=0)  [columnas — cache-unfriendly]", suma_columnas, M_c)
print(f"\n  Ratio: {t_cols/t_filas:.2f}x — las columnas son más lentas en C-order")

In [ ]:
# El layout cambia cuál operación es más rápida
# En Fortran-order, la situación se invierte

print(f"Suma sobre matriz {N}x{N} Fortran-order:")
bench("np.sum(M_f, axis=1)  [filas]",    suma_filas,    M_f)
bench("np.sum(M_f, axis=0)  [columnas]", suma_columnas, M_f)

print()
print("Conclusión: el orden de tus operaciones debe coincidir con el layout de tu matriz.")

In [ ]:
# Los strides también explican por qué transponer y luego operar puede ser lento
# Una transpuesta en NumPy NO mueve datos — solo cambia los strides

M = np.random.rand(1000, 1000)
Mt = M.T      # solo cambia strides, mismos datos en memoria

print("M  strides:", M.strides)    # (8000, 8) bytes
print("Mt strides:", Mt.strides)   # (8, 8000) bytes ← ahora los accesos a filas de Mt saltan 8000 bytes
print()
print("M.T es una vista, no una copia — no mueve datos en memoria")
print("np.ascontiguousarray(M.T) sí copia — útil antes de operaciones intensivas sobre Mt")

---
# 5. Vistas vs Copias — No Muevas Datos que No Tienes que Mover

Ya lo vimos en notebooks anteriores, pero en el contexto de HPC el costo es concreto: copiar datos cuesta tiempo y memoria. Las vistas son gratis.

La pregunta siempre es: **¿esta operación me da una vista o una copia?**

Regla general:
- Slicing básico → **vista**
- Fancy indexing, boolean indexing → **copia**
- `reshape` → **vista** si el arreglo es contiguo, **copia** si no
- `flatten()` → siempre **copia**. `ravel()` → **vista** si puede

In [ ]:
# Verificar si algo es una vista: el atributo .base
# Si .base no es None, el arreglo es una vista de .base

original = np.arange(100)

vista   = original[10:50]           # slice → vista
copia   = original[[10, 20, 30]]    # fancy indexing → copia
raveled = original.ravel()          # ravel → vista si es contiguo

print("vista.base is original:  ", vista.base is original)    # True
print("copia.base is original:  ", copia.base is original)    # False
print("raveled.base is original:", raveled.base is original)  # True

In [ ]:
# Benchmark: operar sobre un slice (vista) vs copiar y operar
N = 5_000_000
arr = np.random.rand(N)

def con_vista(a):
    sub = a[N//4 : 3*N//4]    # vista — sin copia
    return np.sum(sub)

def con_copia(a):
    sub = a[N//4 : 3*N//4].copy()   # forzamos copia
    return np.sum(sub)

print("Operar sobre la mitad de un arreglo de 5M:")
t1 = bench("slice (vista, sin copia)", con_vista, arr)
t2 = bench("slice.copy() + operación", con_copia, arr)
print(f"\n  La copia extra cuesta ~{t2-t1:.2f} ms adicionales")

In [ ]:
# Cuánto cuesta en sí hacer una copia en función del tamaño
print("Costo de .copy() por tamaño:")
print(f"  {'N':>10}   {'copy (ms)':>10}   {'MB copiados':>12}")
print("  " + "-" * 40)

for exp in [5, 6, 7, 8]:
    n = 10 ** exp
    a = np.random.rand(n)
    mb = a.nbytes / 1e6

    t0 = time.perf_counter()
    for _ in range(5):
        a.copy()
    t1 = time.perf_counter()
    ms = (t1 - t0) / 5 * 1000

    print(f"  {n:>10,}   {ms:>10.3f}   {mb:>12.1f}")

---
# 6. Operaciones In-Place

Las operaciones in-place modifican el arreglo existente **sin alocar un arreglo nuevo**. Esto ahorra tiempo de alocación y presión sobre el garbage collector.

- `a += b` → in-place (modifica `a` directamente)
- `a = a + b` → crea un arreglo temporal nuevo para `a + b`, luego lo asigna a `a`

La diferencia parece pequeña en papel, pero en arreglos grandes es una alocación de memoria extra que paga el tiempo de `malloc` y después tiene que ser recolectada.

In [ ]:
N = 10_000_000

def no_inplace(n):
    a = np.ones(n)
    a = a + 1.0    # crea arreglo temporal, luego reasigna
    a = a * 2.0
    return a

def inplace(n):
    a = np.ones(n)
    a += 1.0       # modifica en el buffer existente
    a *= 2.0
    return a

print("Operaciones aritméticas en arreglo de 10M elementos:")
t1 = bench("a = a + 1; a = a * 2  (no in-place)", no_inplace, N)
t2 = bench("a += 1; a *= 2        (in-place)",    inplace,    N)
print(f"\n  Speedup: ~{t1/t2:.2f}x")

In [ ]:
# NumPy también tiene versiones explícitas in-place de las ufuncs
# con el parámetro `out=`

a = np.random.rand(5_000_000)
out = np.empty_like(a)   # buffer preasignado

def sqrt_nuevo(arr):
    return np.sqrt(arr)          # aloca nuevo arreglo

def sqrt_out(arr, buffer):
    np.sqrt(arr, out=buffer)     # escribe en buffer existente
    return buffer

print("np.sqrt en 5M elementos:")
t1 = bench("np.sqrt(arr)",          sqrt_nuevo, a)
t2 = bench("np.sqrt(arr, out=buf)", sqrt_out,   a, out)
print(f"\n  Speedup con out=: ~{t1/t2:.2f}x")

### ⚠️ In-place modifica el arreglo original

Si tienes una vista de un arreglo y haces operaciones in-place sobre ella, el original también cambia. Asegúrate de que eso es lo que quieres antes de usar `+=` o `*=`.

In [ ]:
original = np.array([1.0, 2.0, 3.0, 4.0])
vista = original[:2]

vista += 100   # in-place sobre la vista → modifica original también
print(original)   # [101., 102.,   3.,   4.]

---
# 7. Selección de dtype

El dtype determina cuántos bytes ocupa cada elemento. Menos bytes = menos memoria = menos datos que mover entre RAM y CPU cache = más velocidad.

En muchas aplicaciones (machine learning, procesamiento de imágenes, simulaciones) **no necesitas float64**. `float32` tiene la mitad del tamaño y en operaciones vectorizadas puede ser igual o más rápido porque caben el doble de elementos en cada cache line.

In [ ]:
N = 10_000_000
a64 = np.random.rand(N).astype(np.float64)   # 8 bytes/elemento → 80 MB
a32 = a64.astype(np.float32)                 # 4 bytes/elemento → 40 MB

print(f"float64: {a64.nbytes / 1e6:.0f} MB")
print(f"float32: {a32.nbytes / 1e6:.0f} MB")
print()

print("np.sum en 10M elementos:")
t64 = bench("float64", np.sum, a64)
t32 = bench("float32", np.sum, a32)
print(f"\n  float32 es ~{t64/t32:.2f}x más rápido para sum")

In [ ]:
# Lo mismo para multiplicación matricial — donde el impacto es mayor
N = 1000
A64 = np.random.rand(N, N).astype(np.float64)
A32 = A64.astype(np.float32)

print(f"Multiplicación matricial {N}x{N}:")
t64 = bench("float64 @ float64", np.matmul, A64, A64)
t32 = bench("float32 @ float32", np.matmul, A32, A32)
print(f"\n  float32 es ~{t64/t32:.2f}x más rápido para matmul")

In [ ]:
# Resumen de dtypes útiles y cuándo usarlos
dtypes = [
    ("float64", np.float64, "Cálculo científico, cuando necesitas precisión máxima"),
    ("float32", np.float32, "ML/DL, GPUs, cuando el rango de float64 es excesivo"),
    ("float16", np.float16, "Inferencia en GPU, muy limitado en rango y precisión"),
    ("int64",   np.int64,   "Enteros grandes, índices grandes"),
    ("int32",   np.int32,   "Enteros medianos, más eficiente en memoria"),
    ("int8",    np.int8,    "Rango -128 a 127, procesamiento de imágenes"),
    ("uint8",   np.uint8,   "Rango 0-255, canales de color en imágenes"),
    ("bool",    np.bool_,   "Máscaras booleanas"),
]

print(f"{'dtype':<10} {'bits':>5} {'bytes':>6}   uso típico")
print("-" * 65)
for nombre, dtype, uso in dtypes:
    item = np.zeros(1, dtype=dtype)
    print(f"{nombre:<10} {item.itemsize*8:>5} {item.itemsize:>6}   {uso}")

---
# 8. Preasignación de Memoria

Uno de los errores más comunes en código numérico es ir creciendo arreglos dinámicamente con `np.append` dentro de un loop. `np.append` **siempre crea un arreglo nuevo** — es una copia completa cada vez. En un loop de N iteraciones, eso es $O(N^2)$ en tiempo y memoria.

La solución: **preasignar el arreglo de salida al tamaño final** y llenar posiciones.

In [ ]:
N = 50_000

# ❌ Patrón incorrecto: np.append en loop
def crecimiento_dinamico(n):
    resultado = np.array([])
    for i in range(n):
        resultado = np.append(resultado, i * 2.0)   # copia completa cada vez
    return resultado

# ✅ Patrón correcto: preasignar + llenar
def preasignado(n):
    resultado = np.empty(n)      # alocación única
    for i in range(n):
        resultado[i] = i * 2.0   # solo escribe en el buffer
    return resultado

# ✅✅ Mejor aún: vectorizado
def vectorizado(n):
    return np.arange(n) * 2.0

print(f"Construir arreglo de {N:,} elementos:")
t1 = bench("np.append en loop      ❌", crecimiento_dinamico, N, reps=3)
t2 = bench("np.empty + fill loop   ✅", preasignado,          N)
t3 = bench("np.arange * 2.0       ✅✅", vectorizado,          N)

print(f"\n  Preasignado es ~{t1/t2:.0f}x más rápido que append")
print(f"  Vectorizado es ~{t1/t3:.0f}x más rápido que append")

In [ ]:
# Qué pasa cuando sí necesitas acumular resultados de tamaño desconocido:
# usa una lista Python para acumular y convierte al final

def acumular_lista(n):
    acc = []
    for i in range(n):
        if i % 3 == 0:           # tamaño final no conocido de antemano
            acc.append(i * 2.0)
    return np.array(acc)         # conversión única al final

print("\nAcumular resultados de tamaño desconocido:")
bench("lista.append + np.array al final", acumular_lista, N)
print("  (mucho más rápido que np.append en loop)")

---
# 9. `np.einsum` — Operaciones Tensóricas sin Loops ni Transpuestas

`np.einsum` (Einstein summation) es una función que expresa contracciones de tensores con una notación de índices. En términos simples: puedes describir cualquier combinación de multiplicaciones y sumas sobre ejes con un string, y NumPy lo ejecuta de forma optimizada.

Es especialmente útil cuando la operación que necesitas no tiene una función directa en NumPy, o cuando quieres evitar crear arreglos intermedios.

In [ ]:
# La notación de einsum: cada letra es un índice
# La string 'ij,jk->ik' significa:
#   - toma un tensor con índices (i,j) y otro con índices (j,k)
#   - contrae (suma) sobre el índice j
#   - el resultado tiene índices (i,k)
# → eso es multiplicación de matrices

A = np.random.rand(3, 4)
B = np.random.rand(4, 5)

C_matmul = A @ B
C_einsum = np.einsum('ij,jk->ik', A, B)

print("A @ B == np.einsum('ij,jk->ik', A, B):", np.allclose(C_matmul, C_einsum))

In [ ]:
# Otras operaciones que einsum expresa limpiamente

A = np.random.rand(4, 4)
v = np.random.rand(4)

# Traza (suma de la diagonal)
print("Traza:      ", np.einsum('ii->', A), "==", np.trace(A))

# Transpuesta
At = np.einsum('ij->ji', A)
print("Transpuesta igual:", np.allclose(At, A.T))

# Producto externo (outer product) de dos vectores
outer = np.einsum('i,j->ij', v, v)
print("Outer product shape:", outer.shape, "== np.outer:", np.allclose(outer, np.outer(v, v)))

# Suma por filas
print("Suma filas: ", np.einsum('ij->i', A), "==", A.sum(axis=1))

In [ ]:
# Caso donde einsum brilla: batch matrix multiplication
# Multiplicar N matrices de tamaño (m x k) con N matrices de tamaño (k x p)
# sin escribir ningún loop

batch = 64
m, k, p = 32, 128, 32

A_batch = np.random.rand(batch, m, k)
B_batch = np.random.rand(batch, k, p)

# Con loop — la forma obvia
def batch_loop(A, B):
    resultado = np.empty((A.shape[0], A.shape[1], B.shape[2]))
    for i in range(A.shape[0]):
        resultado[i] = A[i] @ B[i]
    return resultado

# Con einsum — sin loop
def batch_einsum(A, B):
    return np.einsum('bik,bkj->bij', A, B)

# Con matmul — NumPy 1.16+ soporta batch matmul directamente
def batch_matmul(A, B):
    return np.matmul(A, B)

print(f"Batch matmul: {batch} matrices de ({m}x{k}) @ ({k}x{p}):")
bench("loop Python + @",     batch_loop,   A_batch, B_batch)
bench("np.einsum",           batch_einsum, A_batch, B_batch)
bench("np.matmul (batch)",   batch_matmul, A_batch, B_batch)

# Verificamos que dan el mismo resultado
print("\n  Resultados iguales:", np.allclose(batch_loop(A_batch, B_batch),
                                              batch_matmul(A_batch, B_batch)))

In [ ]:
# einsum tiene un modo optimizado que elige el orden de evaluación
# para minimizar operaciones — útil para cadenas de más de dos tensores

A = np.random.rand(100, 200)
B = np.random.rand(200, 300)
C = np.random.rand(300, 50)

# Triple producto matricial A @ B @ C
def triple_default(A, B, C):
    return np.einsum('ij,jk,kl->il', A, B, C)

def triple_optimized(A, B, C):
    return np.einsum('ij,jk,kl->il', A, B, C, optimize=True)  # elige el orden óptimo

def triple_matmul(A, B, C):
    return (A @ B) @ C

print("Triple producto matricial (100x200) @ (200x300) @ (300x50):")
bench("einsum sin optimize",   triple_default,   A, B, C)
bench("einsum optimize=True",  triple_optimized, A, B, C)
bench("(A @ B) @ C",           triple_matmul,    A, B, C)

---
# 10. Conectando con el Hardware: BLAS y Paralelismo Implícito

Cuando llamas `A @ B` para matrices grandes, NumPy no implementa la multiplicación matricial él mismo — la delega a una librería de álgebra lineal optimizada para tu CPU específico: **BLAS** (Basic Linear Algebra Subprograms).

Las implementaciones modernas de BLAS (OpenBLAS, MKL, BLIS) usan:
- **Instrucciones SIMD** (AVX, AVX-512) para procesar múltiples floats por ciclo de reloj
- **Múltiples threads** en paralelo automáticamente
- **Blocking** para mantener datos en L2/L3 cache durante la multiplicación

Todo eso pasa sin que hagas nada — simplemente por usar `@` en lugar de un loop.

In [ ]:
# Verificar qué BLAS está usando NumPy
np.show_config()

In [ ]:
# La multiplicación matricial escala mejor que O(n^3) en la práctica
# porque BLAS usa SIMD y multi-threading

print("Tiempo de matmul por tamaño de matriz (NxN):")
print(f"  {'N':>6}   {'tiempo (ms)':>12}   {'GFLOPS':>8}")
print("  " + "-" * 35)

for n in [128, 256, 512, 1024, 2048]:
    A = np.random.rand(n, n).astype(np.float32)

    t0 = time.perf_counter()
    for _ in range(5):
        _ = A @ A
    t1 = time.perf_counter()

    ms = (t1 - t0) / 5 * 1000
    # FLOPS para matmul NxN: 2*N^3 operaciones
    gflops = (2 * n**3) / (ms / 1000) / 1e9
    print(f"  {n:>6}   {ms:>12.3f}   {gflops:>8.2f}")

In [ ]:
# Comparación final: implementación ingenua O(n^3) vs BLAS
# Solo con n=128 para que el loop no tarde siglos

n = 128
A = np.random.rand(n, n)
B = np.random.rand(n, n)

def matmul_loop(A, B):
    n = A.shape[0]
    C = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            for k in range(n):
                C[i, j] += A[i, k] * B[k, j]
    return C

def matmul_numpy(A, B):
    return A @ B

print(f"Multiplicación matricial {n}x{n}:")
t1 = bench("triple loop Python (O(n^3))", matmul_loop,  A, B, reps=2)
t2 = bench("A @ B (BLAS)",               matmul_numpy, A, B)
print(f"\n  BLAS es ~{t1/t2:.0f}x más rápido — y esto es solo con n={n}")
print(f"  Con n=1024 el triple loop tardaría ~{t1 * (1024/n)**3 / 1000:.0f} segundos")
print(f"  BLAS lo hace en ~{t2 * (1024/n)**3:.1f} ms")

---
# Checklist de Optimización con NumPy

Antes de buscar soluciones más complejas (Numba, Cython, C extensions), revisa esta lista en orden. En la mayoría de los casos, alguna de estas es suficiente.

| # | Técnica | Cuándo aplicarla |
|---|---------|------------------|
| 1 | **Eliminar loops** — vectorizar con operaciones NumPy | Siempre que haya un loop sobre un arreglo |
| 2 | **Usar el dtype correcto** — float32 en lugar de float64 | ML, imágenes, simulaciones donde la precisión extra no importa |
| 3 | **Preasignar memoria** — `np.empty` en lugar de `np.append` | Cuando el tamaño final es conocido |
| 4 | **Operaciones in-place** — `+=`, `*=`, `out=` | Arreglos grandes, pipelines con muchas operaciones |
| 5 | **Preferir vistas sobre copias** — evitar `.copy()` innecesario | Siempre que solo necesites leer un subconjunto |
| 6 | **Respetar el layout de memoria** — operar en el eje contiguo | Operaciones sobre matrices grandes por filas o columnas |
| 7 | **Usar `np.einsum`** — para contracciones tensóricas complejas | Batch ops, productos que requieren arreglos intermedios |
| 8 | **Medir antes de optimizar** — `%timeit`, `perf_counter` | Siempre, antes de tocar cualquier cosa |

El siguiente nivel son herramientas como **Numba** (JIT compilation), **CuPy** (NumPy en GPU) y **Dask** (NumPy distribuido), CUDA (Compute Unified Device Architecture), que parten de exactamente estos mismos principios.